# Multi-timestep cloud-flux optimization

This notebook is the experiment runner. Reusable SolarPILOT data generation lives in `cloud_flux_data.py`; the JAX/MMA model and plots live in `flux_optimizer.py`. Run the sections in order.

## 1. Configure the experiment

Set every experiment input here. Generated timestep CSVs are placed under `outputs/cloud_flux/` and are intentionally not committed.

In [ ]:
from pathlib import Path
import sys

sys.path.append('./transient/')

from cloud_flux_data import FluxExperimentConfig, generate_timestep_data
from flux_optimizer import HeliostatFluxOptimizerTimeSeries

CONFIG = FluxExperimentConfig(
    weather_file=Path("climate_files/USA NV Tonopah (TMY2).csv"),
    output_dir=Path("outputs/transient"),
    x_res=54,
    y_res=54,
    rated_power_mw=220,
    tower_height_m=170,
    receiver_diameter_m=17.65,
    receiver_height_m=21.8,
    start_hour=9.0,
    end_hour=15.0,
    n_timesteps=5,
)

TARGET_FLUX = Path("data_dynamic/ideal_flux.csv")
REGENERATE_DATA = True  # Set False to reuse the CSVs already in output_dir.


## 2. Generate or load optical data

SolarPILOT is run once per configured hour. The output is one CSV per timestep containing only the optical quantities needed by the optimizer.

In [ ]:
if REGENERATE_DATA:
    heliostat_csv_files = generate_timestep_data(CONFIG)
else:
    heliostat_csv_files = [CONFIG.output_dir / f"heliostat_t{index:02d}.csv" for index in range(CONFIG.n_timesteps)]
    missing = [path for path in heliostat_csv_files if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"Missing generated timestep files: {missing}")

heliostat_csv_files


## 3. Optimize aimpoints

The objective matches the target flux map over all timesteps while penalizing movement between adjacent timesteps. `move_cap_m=None` disables the optional smooth movement-cap constraint.

In [ ]:
optimizer = HeliostatFluxOptimizerTimeSeries(
    target_flux_path=TARGET_FLUX,
    heliostat_csv_files=heliostat_csv_files,
    receiver_radius_m=CONFIG.receiver_radius_m,
    receiver_height_m=CONFIG.receiver_height_m,
    move_penalty_weight=0.1,
    move_cap_m=None,
    cap_sharpness=50.0,
)

optimized_offsets, final_loss = optimizer.optimize_mma(
    restarts=1,
    max_iterations=10,
)
print(f"Final loss (tracking + movement penalty): {final_loss:.6e}")
optimizer.export_travel_to_csv()


## 4. Inspect results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("./outputs/transient/heliostat_t00_with_travel.csv")
plt.scatter(df["x_normal"], df["amplitude"], c=df["total_travel_m"], cmap="viridis", s=1)
#plt.scatter(df["x_normal"], df["total_travel_m"], cmap="viridis", s=1)
plt.xlabel("X Normal (m)")
plt.ylabel("Amplitude (W/m^2)")
plt.title("Movement vs X Normal and Amplitude")
plt.xlim(-2, 58)


In [ ]:
optimizer.plot_movement_summary()
